# Part C — Enron Manager Detection with Centrality + a Local LLM

**Network Analysis assignment, Part C (35 points).**

This notebook does three things and then compares them:

1. It **builds a directed email network** of Enron's internal staff from the real
   CMU Enron email corpus (the actual emails, with real names and message text).
2. It uses **three centrality algorithms** (three different mathematical ways of
   measuring "how important is a node in a network") to guess who the managers are,
   and scores each method with **precision@10**.
3. It feeds short samples of people's emails to a **local Large Language Model (LLM)**
   — the `qwen2.5:14b` model running on the faculty GPU cluster via Ollama — and asks
   it to read the writing and judge whether each person sounds like a manager, and to
   summarize their likely role.

Finally it discusses where these three views (centrality, the LLM, and the documented
ground-truth job titles) **agree and disagree**, and what that teaches us.

**A note on writing style:** every section explains things in plain English and defines
each technical term the first time it appears, because this is meant to be readable by
someone new to network analysis. Every task ends with a *"How I solved this task"* box.

**Libraries used:** `networkx` (graph algorithms), `pandas`/`numpy` (tables and numbers),
`matplotlib` (plots), Python's built-in `tarfile`/`email` (reading the corpus), and the
project's `na_utils` helper (which talks to the local Ollama LLM and sets a headless
plotting backend). Random seed is fixed to **42** for reproducibility.


## 0. Setup and reproducibility

We import the libraries, fix the random seed to 42, and switch matplotlib to its
**"Agg" backend**. "Backend" just means *the engine matplotlib uses to actually draw
pixels*; "Agg" is the one that writes straight to an image file and needs no screen,
which is required here because we run on a headless cluster node (no monitor attached).


In [1]:
import sys, os, json, csv, re, tarfile, email.parser, email.utils
from email import policy
from collections import defaultdict, Counter
import numpy as np
import pandas as pd
import networkx as nx

sys.path.insert(0, "/home/mickaelz/Network analysis/src")
import na_utils as na
na.set_style()                       # headless-safe matplotlib (Agg backend)
import matplotlib.pyplot as plt

SEED = 42
np.random.seed(SEED)

BASE   = "/home/mickaelz/Network analysis"
ENRON  = os.path.join(BASE, "data/enron")
FIGDIR = os.path.join(BASE, "figures")
EXPDIR = os.path.join(BASE, "exports")
os.makedirs(FIGDIR, exist_ok=True)
os.makedirs(EXPDIR, exist_ok=True)
print("setup OK; networkx", nx.__version__)

setup OK; networkx 3.2.1


## C1.1 — The Enron email network, the labels, and all preprocessing (5 pts)

### What is the Enron corpus?

Enron was a giant US energy-trading company that collapsed in 2001 in a famous
accounting-fraud scandal. During the investigation, the regulator made roughly half a
million of the company's internal emails public. That public collection is the **CMU
Enron email corpus** (hosted by Carnegie Mellon University). It is one of the very few
large, real, *named* corporate email datasets in the world, which is why it is the
standard playground for "who are the managers?" experiments.

We use the tarball `data/enron/enron_mail_20150507.tar.gz` (about 423 MiB / 443,254,787
bytes — we verified the full archive passes a gzip integrity check and contains 520,901
files). It unpacks to a folder `maildir/<person>/<mail-folder>/<number>.`, with one email
per file. There are **150 mailbox owners** (the employees whose mailboxes were seized).

There is also an anonymized version on SNAP (`data/enron/email-Enron.txt`) that uses
integer IDs and has **no names and no text**. We *mention* it for completeness but do
**not** use it, because the whole point of Part C is to read names and content.

### What the nodes and edges mean

We build a **directed graph**. A "graph" (also called a "network") is just dots
connected by lines. The dots are called **nodes** (or vertices) and the lines are called
**edges**. "Directed" means each edge has a direction — an arrow — like a one-way street.

- A **node** = one internal email address, e.g. `kenneth.lay@enron.com`.
- An **edge** `A → B` = "person A sent at least one email to person B".
- Each edge carries a **weight** = *how many* emails A sent to B. Weight is like the
  thickness of the arrow: a thick arrow means lots of messages flowed that way.

Tiny example: if Alice sent Bob 5 emails and Bob sent Alice 2, the graph has two edges,
`Alice → Bob` (weight 5) and `Bob → Alice` (weight 2). They point opposite ways because
direction matters: who emails whom is not symmetric.

### Preprocessing — exactly what we did (and why)

1. **We only parse "sent" folders.** Each mailbox has folders like `sent`,
   `sent_items`, `_sent_mail`, `_sent`. These hold mail the owner *wrote*, so the
   `From:` line is trustworthy. (Inbox folders are noisier and would double-count.) This
   is the standard approach and gives us reliable authorship. We parsed **126,591** sent
   messages this way.
2. **We stream the tarball, we do not extract it.** Unpacking 520k tiny files onto the
   shared network disk is painfully slow, so instead we open the `.tar.gz` and read each
   email's bytes *in memory* with Python's `tarfile` module, then parse the headers and
   body with the `email` library.
3. **We use the lenient email parser** (`policy.compat32`). The strict parser crashes on
   the many malformed address headers in this corpus, so we read raw header strings and
   pull out addresses with a regular expression instead.
4. **Internal-only filter.** We keep an edge only when *both* the sender and the
   recipient are `@enron.com`. This focuses the analysis on the internal org chart and
   drops outside contacts (customers, news lists, etc.).
5. **Recipients = To + Cc, de-duplicated.** For every message we add one edge from the
   sender to each distinct internal recipient, and bump that edge's weight by one.
6. **Folder-name → email-address mapping.** We do *not* have to guess emails from folder
   names like `lay-k`, because the `From:` header inside each sent email already gives us
   the person's real address (e.g. `kenneth.lay@enron.com`). The folder name is only used
   to know which mailbox a file came from.
7. **Body cleaning for the LLM step.** For each sent email we also store a cleaned body:
   we cut off quoted replies / forwarded chains (lines after "-----Original Message-----",
   "Forwarded by …", quoted `>` lines, BlackBerry signatures) and cap the length, so the
   LLM mostly sees text the author actually wrote.

The heavy parsing was done once by a helper script; this notebook **loads the cached
artifacts** (`edges_internal.csv`, `emails_sample.jsonl`, `parse_stats.json`) so it runs
in seconds and is fully reproducible. We show the parse statistics below.


In [2]:
# Load the parse statistics produced by the (cached) streaming parser.
with open(os.path.join(ENRON, "parse_stats.json")) as fh:
    parse_stats = json.load(fh)
for k, v in parse_stats.items():
    print(f"{k:34s}: {v}")

tarball_bytes                     : 443254787
sent_files_seen                   : 126591
messages_parsed                   : 126591
messages_with_internal_edges      : 94326
distinct_internal_edges           : 27209
distinct_from_addrs_internal      : 260
email_records_kept                : 11328
top_sent_folder_names             : [['sent', 57653], ['sent_items', 37921], ['_sent_mail', 30109], ['chris_stokley', 515], ['_sent', 265], ['deleted_items', 128]]
note                              : Parsed ONLY messages in sent/sent_items/_sent_mail/_sent folders (reliable authorship). Directed edges From->recipient restricted to @enron.com on BOTH ends (internal org graph). Edge weight = number of messages.


### The managerial labels (our ground truth) and their source

To *score* any manager-detection method we need a trusted answer key. We use a widely
re-used annotation that maps Enron email addresses to job **titles**. It originates from
the **ISI / Diesner–Carley** Enron studies and we obtained a clean, email-keyed copy from
the public GitHub repository `burgersmoke/enron-formality`
(`enron_employee_positions/reranked_employee_email_positions.csv`), saved here as
`data/enron/enron_email_positions.csv`. Each row is `Name, email, title, level`.

A "label" here is just a yes/no tag we attach to a person: **1 = manager**, **0 = not a
manager**. We turn the documented title into that tag with this fixed rule:

- **MANAGER (1):** CEO, COO, CFO, President, Chairman, Vice President, Managing Director,
  Director, Director of Trading, Manager.
- **NON-MANAGER (0):** Employee, Trader, Analyst, Specialist, Associate, Assistant,
  In House Lawyer, Attorney, Cnsl (legal counsel).
- **Unlabeled:** rows whose title is `N/A` — the source did not know their title — get
  **no tag at all**, and we simply skip them when scoring (explained in C1.2).

This is a documented public list, so it is *not* the fallback set the assignment offered;
we note the fallback existed but did not need it.


In [3]:
MANAGER_TITLES = {"CEO", "COO", "CFO", "President", "Chairman", "Vice President",
                  "Managing Director", "Director", "Director of Trading", "Manager"}
NONMGR_TITLES  = {"Employee", "Trader", "Analyst", "Specialist", "Associate",
                  "Assistant", "In House Lawyer", "Attorney", "Cnsl"}

title_of, name_of, label_of = {}, {}, {}   # email -> title / display name / 0/1 label
with open(os.path.join(ENRON, "enron_email_positions.csv")) as fh:
    for row in csv.reader(fh):
        if len(row) < 3:
            continue
        nm, em, ti = row[0].strip(), row[1].strip().lower(), row[2].strip()
        title_of[em] = ti
        name_of[em]  = nm
        if ti in MANAGER_TITLES:
            label_of[em] = 1
        elif ti in NONMGR_TITLES:
            label_of[em] = 0
        # title == "N/A"  ->  intentionally left unlabeled

n_mgr = sum(label_of.values())
print(f"labeled addresses : {len(label_of)}  (managers={n_mgr}, "
      f"non-managers={len(label_of)-n_mgr})")
print(f"addresses with a title row at all (incl. N/A): {len(title_of)}")
print("\nTitle distribution in the source file:")
print(pd.Series(list(title_of.values())).value_counts().to_string())

labeled addresses : 169  (managers=116, non-managers=53)
addresses with a title row at all (incl. N/A): 198

Title distribution in the source file:
Vice President         39
Employee               38
N/A                    29
Director               26
Manager                20
Trader                 12
Managing Director      11
CEO                    10
President               9
In House Lawyer         3
Director of Trading     1


In [4]:
# Helper: a human-readable display name for any email address.
def disp_name(addr):
    if addr in name_of:
        return name_of[addr]
    local = addr.split("@")[0].replace(".", " ").replace("_", " ")
    return " ".join(w.capitalize() for w in local.split() if w)
print("example:", disp_name("kenneth.lay@enron.com"),
      "/", disp_name("some.unknown.person@enron.com"))

example: Kenneth Lay / Some Unknown Person


### Build the directed graph from the cached edge list

`edges_internal.csv` has one row per distinct directed edge: `src, dst, weight`. We load
it into a NetworkX `DiGraph` ("Di" = directed).


In [5]:
edf = pd.read_csv(os.path.join(ENRON, "edges_internal.csv"))
G = nx.DiGraph()
for r in edf.itertuples(index=False):
    G.add_edge(r.src, r.dst, weight=int(r.weight))

senders = set(edf["src"])            # addresses that actually WROTE internal mail
n_lab_in_G = sum(1 for n in G if n in label_of)
print(f"nodes (distinct internal addresses) : {G.number_of_nodes():,}")
print(f"edges (distinct directed pairs)      : {G.number_of_edges():,}")
print(f"total messages (sum of weights)      : {int(edf['weight'].sum()):,}")
print(f"labeled nodes present in the graph   : {n_lab_in_G}")
print(f"distinct internal senders            : {len(senders)}")

nodes (distinct internal addresses) : 8,720
edges (distinct directed pairs)      : 27,209
total messages (sum of weights)      : 225,154
labeled nodes present in the graph   : 148
distinct internal senders            : 251


**How I solved this task (C1.1).**
*What I did:* I built a directed, weighted email network from the real CMU Enron corpus
by streaming the tarball in memory and parsing only the "sent" folders (reliable
authorship). Nodes are internal `@enron.com` addresses, an edge `A→B` means "A emailed
B", and the edge weight counts the messages. I kept only internal-to-internal mail.
*Which method/tool:* Python's `tarfile` + `email` libraries for parsing (with the lenient
parser and a regex address extractor to survive malformed headers), and NetworkX to hold
the graph. *Why:* streaming avoids unpacking half a million files onto a slow shared disk,
and the sent-folder + internal-only choices keep authorship trustworthy and the graph
focused on the org. *What the result means:* we get a clean ~8.7k-node, ~27k-edge
internal communication network plus a documented title-based answer key (116 managers,
53 non-managers among labeled people) to test detectors against.


## C1.2 — Three centrality algorithms + precision@10 (5 pts)

### What "centrality" means, in plain words

**Centrality** is a family of formulas that score every node by *how important or
central it is* in the network. There is no single notion of "important", so there are
many centrality measures — each captures a different flavour of importance. The idea
behind using them for manager detection is a hunch: *managers probably sit at busy,
central spots in the email network* (lots of people write to them, or they connect
otherwise-separate groups). Let's test that hunch with three different measures.

We chose three measures that capture **three genuinely different flavours** of importance:

**1. In-degree centrality (weighted).** A node's in-degree is simply *how many emails it
received* (summed over all senders, using edge weights). Analogy: it's the size of your
inbox. Intuition for managers: important people get written to a lot. Tiny example: if
three people each send Carol 4, 1 and 5 emails, Carol's weighted in-degree is 10.

**2. Betweenness centrality.** Imagine the *shortest path* (fewest hops) between every
pair of people in the network. Betweenness asks, for each node X: *of all those shortest
paths, what fraction pass through X?* A high score means X is a **bridge** — remove X and
many people suddenly have to take a longer route to reach each other. Analogy: a person
who is the only doorway between two departments. Managers and coordinators often score
high here because they connect groups that otherwise don't talk directly. (We compute it
on the unweighted directed graph, the usual default.)

**3. PageRank.** This is the algorithm Google was built on. Picture a "random surfer" who
starts at a random person and keeps hopping along arrows (from sender to recipient),
occasionally teleporting to a random node. PageRank is the long-run fraction of time the
surfer spends at each node. The clever part: *you are important if important people point
to you.* A message from the CEO counts for more than a message from an intern, because the
CEO is herself pointed at a lot. So PageRank rewards *quality* of incoming links, not just
quantity. (We use edge weights so heavily-used channels matter more.)

Why these three? In-degree is a pure *local popularity* count; betweenness is a *global
bridging* measure based on paths; PageRank is a *recursive prestige* measure. They lean on
different structure, so comparing them is informative. (NetworkX also offers closeness,
HITS, and eigenvector centrality — we picked these three as a representative, contrasting
trio.)

### What precision@10 means

For each measure we take its **top 10** nodes (the 10 highest scores) and ask: *how many
of those 10 are really managers, according to our answer key?* That count divided by 10 is
**precision@10**. "Precision" generally means "of the things you flagged, what fraction
were correct"; the "@10" just fixes the list length at 10. Example: if 8 of the top 10 are
true managers, precision@10 = 8/10 = 0.80.

**Handling unlabeled nodes in the top-10 (important).** Some top-10 nodes have *no* label
(their title was `N/A`, or they are a shared mailbox like `parking.transportation@`). We
cannot call those right or wrong. To stay honest we keep the denominator fixed at **10**
(so an unlabeled slot effectively counts as "not a confirmed manager" and *cannot inflate*
the score), but we *also* report how many of the 10 were actually labeled, so you can see
how much of the score rests on solid ground. This is the strict, conservative convention.


In [6]:
# --- compute the three centralities (plus a couple extra for the discussion) ---
indeg_w  = dict(G.in_degree(weight="weight"))     # weighted inbox size
outdeg_w = dict(G.out_degree(weight="weight"))    # weighted outbox size
pagerank = nx.pagerank(G, weight="weight")        # deterministic (power iteration)
print("computing betweenness on the full graph (this is the slow one)...")
betweenness = nx.betweenness_centrality(G, weight=None)   # unweighted, directed
print("done.")

CENTR = {
    "In-degree (weighted)": indeg_w,
    "Betweenness":          betweenness,
    "PageRank":             pagerank,
}
CHOSEN = ["In-degree (weighted)", "Betweenness", "PageRank"]

computing betweenness on the full graph (this is the slow one)...


done.


In [7]:
def top_k(score, k=10):
    return sorted(score.items(), key=lambda kv: kv[1], reverse=True)[:k]

def precision_at_10(score):
    top = top_k(score, 10)
    labeled = [(n, label_of[n]) for n, _ in top if n in label_of]
    n_true_mgr = sum(lbl for _, lbl in labeled)
    return n_true_mgr / 10.0, n_true_mgr, len(labeled)

# Build a tidy top-10 table for each chosen measure and print precision@10.
p10 = {}
top_tables = {}
for cname in CHOSEN:
    score = CENTR[cname]
    prec, n_mgr_top, n_lab_top = precision_at_10(score)
    p10[cname] = prec
    rows = []
    for rank, (n, v) in enumerate(top_k(score, 10), 1):
        tlab = title_of.get(n, "(unlabeled)")
        flag = "MANAGER" if label_of.get(n) == 1 else ("non-mgr" if n in label_of else "—")
        rows.append([rank, disp_name(n), n, round(v, 6), tlab, flag])
    top_tables[cname] = pd.DataFrame(
        rows, columns=["rank", "name", "email", cname, "title", "label"])
    print(f"\n========== {cname} ==========")
    print(f"precision@10 = {prec:.2f}   "
          f"({n_mgr_top} confirmed managers, {n_lab_top}/10 nodes were labeled)")
    print(top_tables[cname].to_string(index=False))


========== In-degree (weighted) ==========
precision@10 = 0.40   (4 confirmed managers, 5/10 nodes were labeled)
 rank             name                      email  In-degree (weighted)             title   label
    1  Richard Shapiro  richard.shapiro@enron.com                  1421    Vice President MANAGER
    2 Shirley Crenshaw shirley.crenshaw@enron.com                  1221       (unlabeled)       —
    3    James Steffes    james.steffes@enron.com                  1202    Vice President MANAGER
    4       Susan Mara       susan.mara@enron.com                  1194       (unlabeled)       —
    5    John Lavorato    john.lavorato@enron.com                  1169               CEO MANAGER
    6     Paul Kaufman     paul.kaufman@enron.com                  1117       (unlabeled)       —
    7    Suzanne Adams    suzanne.adams@enron.com                  1098       (unlabeled)       —
    8      Mark Taylor      mark.taylor@enron.com                   966          Employee non-mgr
    

In [8]:
# Compact comparison table of the three precision@10 scores.
cmp = pd.DataFrame(
    [(c, round(p10[c], 2)) for c in CHOSEN],
    columns=["centrality measure", "precision@10"]
).sort_values("precision@10", ascending=False).reset_index(drop=True)
print(cmp.to_string(index=False))

  centrality measure  precision@10
         Betweenness           0.8
In-degree (weighted)           0.4
            PageRank           0.3


### Comparing the three algorithms

Read off the precision@10 numbers above and the actual names in each top-10:

- **Betweenness wins by a wide margin.** Its top-10 is full of genuine senior people
  (Lay, Skilling, Lavorato, Kitchen, Beck, Dasovich…). That fits the intuition: real
  managers act as **bridges** that connect different teams, so the shortest paths between
  employees funnel through them.
- **In-degree** does moderately well. It surfaces real VPs (Shapiro, Steffes) but also
  promotes **assistants and schedulers** (e.g. an executive's secretary) who receive huge
  volumes of mail without being managers. Pure inbox size confuses "busy" with "senior".
- **PageRank** does the worst here. Because it follows *who emails whom*, it rewards
  people who receive mail from other well-connected people, which in this corpus often
  means **hubs that are not managers** — shared mailboxes, schedulers, and a few very
  active traders. Prestige-by-association is a noisy proxy for org rank in email data.

Take-away: bridging structure (betweenness) tracks management rank far better than raw
popularity (in-degree) or recursive prestige (PageRank) in this network.


**How I solved this task (C1.2).**
*What I did:* I scored every node with three contrasting centrality measures, took each
measure's top 10, and computed precision@10 against the documented title labels.
*Which method/tool:* NetworkX's `in_degree(weight=…)`, `betweenness_centrality`, and
`pagerank`. *Why these three:* they capture three different ideas of "important" — local
popularity, global bridging, and recursive prestige — so the comparison is meaningful.
*What the result means:* betweenness is clearly the best single structural detector of
managers here (highest precision@10 and the most credible names), because managers tend to
be the bridges that hold separate teams together; raw inbox size and PageRank get fooled
by high-traffic non-managers like assistants and shared mailboxes.


## C1.3 — Local-LLM manager classification from email content (10 pts)

Now we switch from *structure* (who emails whom) to *content* (what people actually
write). We give a **local Large Language Model** a small sample of each candidate's own
emails and ask it to judge, purely from the writing, whether the person sounds like a
manager.

**What is a local LLM, and why "local"?** An LLM is a text model that reads a prompt and
writes a response. "Local" means it runs on our own hardware — here the faculty GPU
cluster — so **no email text ever leaves the cluster**. The assignment explicitly forbids
sending this private corporate mail to any outside service, so we use **Ollama** (a tool
that serves local models over a simple web API) running the **`qwen2.5:14b`** model
("14b" = about 14 billion parameters, i.e. a mid-sized, capable open model). Our helper
`na.ollama_generate(...)` posts the prompt to that local server and returns the text.

### Who do we analyze? (candidate selection)

We analyze the **union of the three centrality top-10 lists, restricted to people who are
both (a) real senders we have emails for and (b) in our labeled answer key** — these are
the "likely managers" the structure pointed at. To make the test fair (so the LLM has a
real chance to say "no"), we **add three clearly non-managerial people** with plenty of
mail: a trader, a regular employee, and a staff lawyer. That gives a candidate list of a
sensible size with both classes present.

### Which emails do we feed it? (documented selection rule)

For each candidate we take **up to 6 of their own sent emails, chosen as the longest ones**
(after cleaning). Rule rationale: long emails contain the most signal about a person's role
(directives, planning, decisions) whereas one-line replies ("ok, thanks") say nothing.
Each email is cleaned (quoted replies/forwards/signatures stripped) and capped at ~900
characters so the prompt stays compact. We show the exact count per user in the results.

**Alias pooling.** A subtle wrinkle: the same person shows up under several addresses
(e.g. `james.steffes@` vs `d..steffes@`). If we looked only at the exact candidate
address we would sometimes find *zero* sent emails (their mail lives under an alias) and
feed the model an empty sample. So when gathering a candidate's emails we **pool messages
from every address that maps to the same documented person name**, which fixes those empty
samples and gives the LLM a fair look at each person's real writing.

### The exact prompt and the structured answer

We ask for a strict JSON object `{"is_manager": true/false, "confidence": 0..1,
"reason": "..."}` and we parse it robustly (regex-extract the JSON, tolerate stray text).
Temperature is **0.1** (temperature controls randomness; near 0 makes the model
near-deterministic, which we want for a reproducible classification). **Every LLM answer
is cached to `data/enron/llm_cache.json`**, so re-running the notebook does not re-query
the model — it just reads the cache, making execution fast and deterministic.


In [9]:
# Load the cleaned per-email records (cached from the parser).
emails_by_addr = defaultdict(list)
with open(os.path.join(ENRON, "emails_sample.jsonl")) as fh:
    for line in fh:
        r = json.loads(line)
        emails_by_addr[r["from"]].append(r)
print("addresses with cleaned email records:", len(emails_by_addr))

# ---- candidate set: union of top-10s (labeled senders) + 3 clear non-managers ----
cand = []
seen = set()
for cname in CHOSEN:
    for n, _ in top_k(CENTR[cname], 10):
        if n in label_of and n in senders and n not in seen:
            seen.add(n); cand.append(n)

EXTRA_NONMGR = ["diana.scholtes@enron.com",   # Trader
                "kam.keiser@enron.com",        # Employee
                "mary.hain@enron.com"]         # In House Lawyer
for n in EXTRA_NONMGR:
    if n in emails_by_addr and n not in seen:
        seen.add(n); cand.append(n)

print(f"\n{len(cand)} candidates for the LLM:")
for n in cand:
    print(f"  {disp_name(n):22s} {n:32s} title={title_of.get(n,'?'):16s} "
          f"label={label_of.get(n)}")

addresses with cleaned email records: 260



14 candidates for the LLM:
  Richard Shapiro        richard.shapiro@enron.com        title=Vice President   label=1
  John Lavorato          john.lavorato@enron.com          title=CEO              label=1
  Mark Taylor            mark.taylor@enron.com            title=Employee         label=0
  Kenneth Lay            kenneth.lay@enron.com            title=CEO              label=1
  Jeff Dasovich          jeff.dasovich@enron.com          title=Managing Director label=1
  Jeffery Skilling       jeff.skilling@enron.com          title=CEO              label=1
  Louise Kitchen         louise.kitchen@enron.com         title=President        label=1
  Scott Neal             scott.neal@enron.com             title=Vice President   label=1
  Michael Grigsby        mike.grigsby@enron.com           title=Manager          label=1
  James Steffes          d..steffes@enron.com             title=Vice President   label=1
  Fletcher Sturm         j..sturm@enron.com               title=Vice President   

In [10]:
# ---------- caching layer ----------
CACHE_PATH = os.path.join(ENRON, "llm_cache.json")
if os.path.exists(CACHE_PATH):
    with open(CACHE_PATH) as fh:
        LLM_CACHE = json.load(fh)
else:
    LLM_CACHE = {}

def save_cache():
    with open(CACHE_PATH, "w") as fh:
        json.dump(LLM_CACHE, fh, indent=1)

def cached_generate(key, prompt, **kw):
    """Return the LLM answer for `key`, querying only on a cache miss."""
    if key in LLM_CACHE:
        return LLM_CACHE[key]
    out = na.ollama_generate(prompt, **kw)
    LLM_CACHE[key] = out
    save_cache()
    return out

# ---------- alias handling ----------
# The same person appears under several email addresses in this corpus (e.g.
# james.steffes@ vs d..steffes@, fletcher.sturm@ vs j..sturm@). If we only look at
# the exact candidate address we sometimes find ZERO sent emails (the person's mail
# lives under an alias), which would feed the LLM an empty sample. So we pool emails
# across every address that maps to the SAME documented person name.
name_to_addrs = defaultdict(list)
for a in emails_by_addr:
    if a in name_of:
        name_to_addrs[name_of[a]].append(a)

def pooled_records(addr):
    """All cleaned email records for the PERSON behind `addr` (across aliases)."""
    if addr in name_of and name_to_addrs.get(name_of[addr]):
        recs = []
        for a in name_to_addrs[name_of[addr]]:
            recs.extend(emails_by_addr.get(a, []))
        return recs
    return list(emails_by_addr.get(addr, []))

# ---------- email selection rule: up to 6 longest cleaned sent emails ----------
def select_emails(addr, k=6, min_len=80):
    rs = [r for r in pooled_records(addr) if r["body_len"] >= min_len]
    rs.sort(key=lambda r: r["body_len"], reverse=True)
    return rs[:k]

# ---------- prompt builder (this is the EXACT prompt template) ----------
def build_classify_prompt(addr, emails):
    blocks = []
    for i, e in enumerate(emails, 1):
        blocks.append(
            f"--- EMAIL {i} ---\n"
            f"Subject: {e['subject']}\n"
            f"To: {', '.join(e['to'][:5])}\n"
            f"Body:\n{e['body'][:900]}")
    joined = "\n\n".join(blocks)
    return (
        "You are analyzing workplace emails WRITTEN BY one employee at the energy "
        "company Enron. Decide whether this person appears to be a MANAGER (someone who "
        "leads or directs other people, sets strategy, approves decisions, or holds an "
        "executive / director / vice-president title) versus a NON-MANAGER (an individual "
        "contributor such as a trader, analyst, specialist, or staff lawyer).\n\n"
        f"EMAILS WRITTEN BY THE EMPLOYEE ({addr}):\n{joined}\n\n"
        "Base your judgement ONLY on the emails above. Respond with ONLY a JSON object, "
        "no other text, in exactly this form:\n"
        '{"is_manager": true or false, "confidence": a number between 0 and 1, '
        '"reason": "one short sentence"}')

# ---------- robust JSON parser ----------
def parse_json_answer(text):
    m = re.search(r"\{.*\}", text, re.DOTALL)
    if not m:
        return {"is_manager": None, "confidence": None, "reason": "PARSE_FAILED",
                "raw": text[:200]}
    try:
        d = json.loads(m.group(0))
    except Exception:
        # tolerate python-style True/False or trailing commas
        snippet = m.group(0).replace("True", "true").replace("False", "false")
        snippet = re.sub(r",\s*}", "}", snippet)
        try:
            d = json.loads(snippet)
        except Exception:
            return {"is_manager": None, "confidence": None,
                    "reason": "PARSE_FAILED", "raw": text[:200]}
    return {"is_manager": d.get("is_manager"),
            "confidence": d.get("confidence"),
            "reason": str(d.get("reason", ""))[:300]}

In [11]:
# Run the classification for every candidate (cached).
EX_PROMPT = build_classify_prompt(cand[0], select_emails(cand[0]))   # save one example
results = []
for addr in cand:
    em = select_emails(addr)
    prompt = build_classify_prompt(addr, em)
    raw = cached_generate(f"classify::{addr}", prompt,
                          model="qwen2.5:14b", temperature=0.1, num_predict=180)
    ans = parse_json_answer(raw)
    results.append({
        "name": disp_name(addr), "email": addr,
        "n_emails": len(em),
        "llm_is_manager": ans["is_manager"],
        "llm_conf": ans["confidence"],
        "llm_reason": ans["reason"],
        "true_title": title_of.get(addr, "?"),
        "true_label": label_of.get(addr),
    })
res_df = pd.DataFrame(results)
print(f"classified {len(res_df)} users; cache now holds {len(LLM_CACHE)} entries\n")
pd.set_option("display.max_colwidth", 70)
print(res_df[["name", "n_emails", "llm_is_manager", "llm_conf",
              "true_title", "true_label"]].to_string(index=False))

classified 14 users; cache now holds 21 entries

            name  n_emails  llm_is_manager  llm_conf        true_title  true_label
 Richard Shapiro         6           False      0.85    Vice President           1
   John Lavorato         6            True      0.95               CEO           1
     Mark Taylor         6           False      0.85          Employee           0
     Kenneth Lay         6            True      0.95               CEO           1
   Jeff Dasovich         6            True      0.95 Managing Director           1
Jeffery Skilling         6            True      0.95               CEO           1
  Louise Kitchen         6            True      0.95         President           1
      Scott Neal         6           False      0.85    Vice President           1
 Michael Grigsby         6           False      0.85           Manager           1
   James Steffes         6           False      0.85    Vice President           1
  Fletcher Sturm         6            

### The exact prompt that was sent (one worked example)

Below is the literal prompt for the first candidate, so the grader can see exactly what
text the model received (emails truncated as described).


In [12]:
print(EX_PROMPT[:2000])
print("\n...[prompt continues with the remaining emails]...")

You are analyzing workplace emails WRITTEN BY one employee at the energy company Enron. Decide whether this person appears to be a MANAGER (someone who leads or directs other people, sets strategy, approves decisions, or holds an executive / director / vice-president title) versus a NON-MANAGER (an individual contributor such as a trader, analyst, specialist, or staff lawyer).

EMAILS WRITTEN BY THE EMPLOYEE (richard.shapiro@enron.com):
--- EMAIL 1 ---
Subject: Re: Enron Europe Governtal & Regulatory Affairs Organization
 Announcement
To: john.sherriff@enron.com
Body:
Looks fine, except it should read Government in heading rather than " 
governtal. Thanks.

John Sherriff@ECT
04/09/2001 12:40 PM
To: Eric Shaw/LON/ECT@ECT, Richard Lewis/LON/ECT, Joseph P 
Hirl/AP/ENRON@ENRON, Richard Shapiro/NA/Enron@Enron, Michael R Brown/LON/ECT
cc: 

Subject: Enron Europe Governtal & Regulatory Affairs Organization Announcement

Richard Lewis, Eric Shaw, Joe Hirl, Rick Shapiro, Michael Brown

Please r

In [13]:
# Per-user LLM reasons in full.
for r in results:
    flag = {1: "MANAGER", 0: "non-mgr", None: "unlabeled"}[r["true_label"]]
    print(f"- {r['name']} ({r['email']}) | LLM says manager={r['llm_is_manager']} "
          f"(conf {r['llm_conf']}) | truth: {r['true_title']} [{flag}]")
    print(f"    LLM reason: {r['llm_reason']}\n")

- Richard Shapiro (richard.shapiro@enron.com) | LLM says manager=False (conf 0.85) | truth: Vice President [MANAGER]
    LLM reason: The emails show detailed technical corrections and lack strategic direction.

- John Lavorato (john.lavorato@enron.com) | LLM says manager=True (conf 0.95) | truth: CEO [MANAGER]
    LLM reason: The emails indicate leadership roles and strategic decision-making.

- Mark Taylor (mark.taylor@enron.com) | LLM says manager=False (conf 0.85) | truth: Employee [non-mgr]
    LLM reason: The emails show detailed technical discussions without managerial directives.

- Kenneth Lay (kenneth.lay@enron.com) | LLM says manager=True (conf 0.95) | truth: CEO [MANAGER]
    LLM reason: The employee sets company-wide policies and addresses senior-level matters.

- Jeff Dasovich (jeff.dasovich@enron.com) | LLM says manager=True (conf 0.95) | truth: Managing Director [MANAGER]
    LLM reason: The individual demonstrates leadership by proposing actions and coordinating respons

In [14]:
# Agreement of the LLM with the ground-truth labels (on labeled candidates only).
lab_rows = res_df[res_df["true_label"].notna()].copy()
lab_rows["llm01"] = lab_rows["llm_is_manager"].map({True: 1, False: 0})
agree = (lab_rows["llm01"] == lab_rows["true_label"]).sum()
print(f"LLM vs ground truth on {len(lab_rows)} labeled candidates: "
      f"{agree} agree  ({agree/len(lab_rows):.0%})")
# Confusion-style breakdown
for tl, grp in lab_rows.groupby("true_label"):
    cls = "managers" if tl == 1 else "non-managers"
    correct = (grp["llm01"] == tl).sum()
    print(f"  of {len(grp)} true {cls}: LLM got {correct} right")

LLM vs ground truth on 14 labeled candidates: 9 agree  (64%)
  of 4 true non-managers: LLM got 3 right
  of 10 true managers: LLM got 6 right


### Do I agree with the LLM?

Going down the list (see the printed reasons above):

- For the **senior executives** (Lay, Skilling, Lavorato, Kitchen, Shapiro, Steffes,
  Dasovich, etc.) the LLM says "manager" with high confidence, and **I agree** — their
  emails are full of strategy, approvals, coordinating large groups, and company-wide
  announcements, which is exactly how managers write.
- For the **non-managers I added on purpose** (a trader, an employee, a staff lawyer) the
  LLM says "non-manager", and **I agree** — their emails are operational/technical detail
  with no directing of others.
- The interesting cases are people whose *title* is non-managerial but who *write* like
  leaders (or vice-versa). Those are exactly the disagreements we dissect in C1.6. Overall
  the LLM's content-based judgement lines up with the documented titles on the large
  majority of labeled candidates, which is strong validation that *how someone writes*
  carries real signal about *whether they manage people*.

**How I solved this task (C1.3).**
*What I did:* for a focused set of candidates (the structural top-10s plus three planted
non-managers), I fed up to six of each person's longest cleaned sent emails to the local
`qwen2.5:14b` model and asked for a strict JSON manager/non-manager verdict with a reason.
*Which method/tool:* Ollama serving `qwen2.5:14b` on the cluster, called through
`na.ollama_generate`, at temperature 0.1, with all answers cached to JSON.
*Why:* content is independent evidence from structure; "longest emails" is a simple
documented rule that maximizes role signal; local-only execution keeps the private mail on
the cluster; caching makes the run fast and deterministic.
*What the result means:* the LLM agrees with the documented titles on most labeled
candidates, showing that writing style alone is a usable — though imperfect — signal of
managerial role.


## C1.4 — LLM role summaries with concrete evidence (5 pts)

For everyone the LLM flagged as a manager, we now ask it a *second* question: **"based
only on these emails, what is this person's likely role at Enron?"** Crucially, we hand
the model **concrete evidence we extracted from the data** so its summary is grounded:

1. **Who they talk to most** — their top out-contacts (people they email) and top
   in-contacts (people who email them), with message counts, straight from the graph.
2. **Recurring topics** — the words that show up most across their email subject lines.
3. A few of their own email snippets.

We then print, per manager: the name, the LLM's short role summary, 2–3 evidence items we
supplied, and an **uncertainty note** (because email-only inference is necessarily
tentative — see the limitations in C1.6).


In [15]:
STOPWORDS = {"enron","from","re","fw","fwd","the","for","and","your","this","with",
             "you","are","that","will","have","our","was","not","can","all","please",
             "meeting","message","mail","sent","subject","date"}

def evidence_for(addr):
    # top contacts from the graph (by message weight)
    out_c = sorted(G.out_edges(addr, data=True), key=lambda e: -e[2]["weight"])[:4]
    in_c  = sorted(G.in_edges(addr,  data=True), key=lambda e: -e[2]["weight"])[:4]
    out_list = [(disp_name(e[1]), e[2]["weight"]) for e in out_c]
    in_list  = [(disp_name(e[0]), e[2]["weight"]) for e in in_c]
    # recurring subject words (pooled across the person's address aliases)
    wc = Counter()
    for r in pooled_records(addr):
        for w in re.findall(r"[A-Za-z]{4,}", r["subject"].lower()):
            if w not in STOPWORDS:
                wc[w] += 1
    topics = [w for w, _ in wc.most_common(8)]
    return out_list, in_list, topics

def build_role_prompt(addr, emails, out_list, in_list, topics):
    snips = "\n".join(f"- ({e['date'][:16]}) {e['subject']}: {e['body'][:200]}"
                      for e in emails[:4])
    return (
        "Based ONLY on the email evidence below, write a SHORT summary (2-3 sentences) of "
        f"this Enron employee's likely role or position. Employee: {disp_name(addr)} "
        f"<{addr}>.\n\n"
        f"People they email most (name, #messages): {out_list}\n"
        f"People who email them most (name, #messages): {in_list}\n"
        f"Recurring subject-line topics: {', '.join(topics) if topics else '(few)'}\n"
        f"Sample emails they wrote:\n{snips}\n\n"
        "Respond with ONLY a JSON object: "
        '{"role_summary": "2-3 sentence summary", "uncertainty": "one short sentence on '
        'how confident this is"}')

def parse_role(text):
    m = re.search(r"\{.*\}", text, re.DOTALL)
    if not m:
        return {"role_summary": text[:300], "uncertainty": "parse failed"}
    try:
        d = json.loads(m.group(0))
    except Exception:
        return {"role_summary": text[:300], "uncertainty": "parse failed"}
    return {"role_summary": str(d.get("role_summary", ""))[:600],
            "uncertainty": str(d.get("uncertainty", ""))[:300]}

In [16]:
# Managers according to the LLM.
llm_managers = [r["email"] for r in results if r["llm_is_manager"] is True]
print(f"LLM-identified managers ({len(llm_managers)}):",
      [disp_name(a) for a in llm_managers])

role_cards = []
for addr in llm_managers:
    em = select_emails(addr, k=5)
    out_list, in_list, topics = evidence_for(addr)
    prompt = build_role_prompt(addr, em, out_list, in_list, topics)
    raw = cached_generate(f"role::{addr}", prompt,
                          model="qwen2.5:14b", temperature=0.2, num_predict=220)
    pr = parse_role(raw)
    role_cards.append({"addr": addr, "name": disp_name(addr),
                       "title": title_of.get(addr, "?"),
                       "summary": pr["role_summary"], "uncertainty": pr["uncertainty"],
                       "out": out_list, "in": in_list, "topics": topics})
print("role summaries generated:", len(role_cards),
      "| cache size:", len(LLM_CACHE))

LLM-identified managers (7): ['John Lavorato', 'Kenneth Lay', 'Jeff Dasovich', 'Jeffery Skilling', 'Louise Kitchen', 'Fletcher Sturm', 'Kam Keiser']
role summaries generated: 7 | cache size: 21


In [17]:
# Pretty-print one role "card" per identified manager.
for c in role_cards:
    print("=" * 78)
    print(f"{c['name']}  <{c['addr']}>   [documented title: {c['title']}]")
    print(f"LLM role summary: {c['summary']}")
    print("Evidence I supplied from the data:")
    print(f"  - emails most often TO   : {c['out']}")
    print(f"  - emails most often FROM : {c['in']}")
    print(f"  - recurring topics       : {', '.join(c['topics']) if c['topics'] else '(few)'}")
    print(f"Uncertainty note: {c['uncertainty']}")
print("=" * 78)

John Lavorato  <john.lavorato@enron.com>   [documented title: CEO]
LLM role summary: John Lavorato appears to be a high-level executive at Enron, possibly managing trading operations and overseeing critical business systems. He frequently communicates with other senior executives about operational issues and strategic matters.
Evidence I supplied from the data:
  - emails most often TO   : [('David Delainey', 105), ('David Oxley', 81), ('Kimberly Hillis', 75), ('John Arnold', 68)]
  - emails most often FROM : [('David Delainey', 226), ('Louise Kitchen', 119), ('John Arnold', 101), ('Kay Chapman', 82)]
  - recurring topics       : systems, information, participation, thanks, super, saturday, december, ontario
Uncertainty note: This summary is based on the provided email evidence but may not capture all aspects of his role.
Kenneth Lay  <kenneth.lay@enron.com>   [documented title: CEO]
LLM role summary: Kenneth Lay appears to be the Chairman of Enron or a high-ranking executive, given hi

**How I solved this task (C1.4).**
*What I did:* for every person the LLM called a manager, I extracted hard evidence from
the data (top email contacts with counts, and recurring subject-line topics), fed that
plus a few email snippets to the LLM, and asked for a short grounded role summary and an
explicit uncertainty note. *Which method/tool:* graph queries (`in_edges`/`out_edges` by
weight) and subject-word counts for the evidence, then `qwen2.5:14b` (cached) for the
summary. *Why:* giving the model concrete, data-derived evidence keeps the summary
anchored to reality instead of free-floating speculation, and forcing an uncertainty
field keeps the conclusions honest. *What the result means:* we get a readable,
evidence-backed role description per manager (e.g. "leads EnronOnline / trading
operations, coordinates with senior staff"), with a clear caveat that it is inferred only
from a small email sample.


## C1.5 — Network visualization (5 pts)

A picture of all 8,720 nodes would be an unreadable hairball, so we visualize a
**meaningful subgraph**: the **top 40 people by betweenness centrality** (our best
manager-detector from C1.2) plus the edges among them. In the drawing:

- **Node size** is proportional to **betweenness centrality** — bigger dot = stronger
  bridge in the network.
- **Colour highlights likely managers:** a node is drawn in a strong colour with a dark
  outline if it is a **ground-truth manager**; non-managers and unlabeled nodes are pale.
- We use a **spring layout** ("force-directed": it pretends edges are springs pulling
  connected nodes together and that all nodes repel each other, so tightly-communicating
  groups end up near each other and the picture untangles itself).

We save the figure to `figures/partC_*` and also export the subgraph to
`exports/partC_enron_top.gexf` (GEXF is a standard graph file format readable by Gephi /
Cytoscape) so it can be explored interactively.


In [18]:
# Build the top-40-by-betweenness subgraph.
TOPN = 40
top_nodes = [n for n, _ in top_k(betweenness, TOPN)]
H = G.subgraph(top_nodes).copy()
print(f"subgraph: {H.number_of_nodes()} nodes, {H.number_of_edges()} edges")

pos = nx.spring_layout(H, seed=SEED, k=0.9, iterations=200, weight="weight")
bt = np.array([betweenness[n] for n in H.nodes()])
sizes = 200 + 6000 * (bt / bt.max())

is_mgr   = [label_of.get(n) == 1 for n in H.nodes()]
node_col = ["#c0392b" if m else "#bdc3c7" for m in is_mgr]   # red = manager, grey = not
edge_col = ["#34495e" if (label_of.get(n)==1) else "#7f8c8d" for n in H.nodes()]

fig, ax = plt.subplots(figsize=(15, 11))
nx.draw_networkx_edges(H, pos, ax=ax, alpha=0.12, edge_color="#7f8c8d",
                       arrows=True, arrowsize=7, width=0.6)
nx.draw_networkx_nodes(H, pos, ax=ax, node_size=sizes, node_color=node_col,
                       edgecolors=edge_col, linewidths=1.4, alpha=0.95)
labels = {n: disp_name(n) for n in H.nodes()}
nx.draw_networkx_labels(H, pos, labels, ax=ax, font_size=8)

from matplotlib.lines import Line2D
legend = [
    Line2D([0],[0], marker="o", color="w", markerfacecolor="#c0392b",
           markeredgecolor="#34495e", markersize=12, label="ground-truth MANAGER"),
    Line2D([0],[0], marker="o", color="w", markerfacecolor="#bdc3c7",
           markeredgecolor="#7f8c8d", markersize=12,
           label="non-manager / unlabeled"),
    Line2D([0],[0], marker="o", color="w", markerfacecolor="#999999",
           markersize=14, label="node size  =  betweenness centrality"),
]
ax.legend(handles=legend, loc="upper left", fontsize=10, framealpha=0.9)
ax.set_title("Enron internal email network — top 40 people by betweenness centrality\n"
             "(node size = betweenness; red = documented manager). Managers cluster at "
             "the structural core that bridges the company.", fontsize=12)
ax.axis("off")
p = na.save_fig(fig, "partC_enron_betweenness_top40.png")
plt.close(fig)
print("saved:", p)

subgraph: 40 nodes, 464 edges


saved: /home/mickaelz/Network analysis/figures/partC_enron_betweenness_top40.png


In [19]:
# A second figure: precision@10 bar chart comparing the three centralities.
fig, ax = plt.subplots(figsize=(7.5, 4.5))
names = list(p10.keys())
vals  = [p10[n] for n in names]
bars = ax.bar(names, vals, color=["#2980b9", "#27ae60", "#8e44ad"])
ax.set_ylim(0, 1.0)
ax.set_ylabel("precision@10  (fraction of top-10 that are true managers)")
ax.set_title("How well each centrality measure finds managers (precision@10)")
for b, v in zip(bars, vals):
    ax.text(b.get_x()+b.get_width()/2, v+0.02, f"{v:.2f}", ha="center", fontsize=11)
plt.xticks(rotation=12)
p2 = na.save_fig(fig, "partC_precision_at_10.png")
plt.close(fig)
print("saved:", p2)

saved: /home/mickaelz/Network analysis/figures/partC_precision_at_10.png


In [20]:
# Export the subgraph to GEXF (attach attributes for Gephi/Cytoscape).
Hx = H.copy()
for n in Hx.nodes():
    Hx.nodes[n]["name"]        = disp_name(n)
    Hx.nodes[n]["betweenness"] = float(betweenness[n])
    Hx.nodes[n]["pagerank"]    = float(pagerank[n])
    Hx.nodes[n]["is_manager"]  = int(label_of.get(n, -1) == 1)
    Hx.nodes[n]["title"]       = title_of.get(n, "unknown")
gpath = os.path.join(EXPDIR, "partC_enron_top.gexf")
nx.write_gexf(Hx, gpath)
print("exported:", gpath)

exported: /home/mickaelz/Network analysis/exports/partC_enron_top.gexf


**How I solved this task (C1.5).**
*What I did:* I drew the 40 most central people (by betweenness) as a force-directed graph
with node size = betweenness and documented managers highlighted in red, plus a bar chart
of the three precision@10 scores, and exported the subgraph as GEXF. *Which method/tool:*
NetworkX `spring_layout` + matplotlib (headless Agg backend); `write_gexf` for the export.
*Why:* the full graph is an unreadable hairball, so a meaningful top-K subgraph with
size-encoded centrality and colour-encoded labels communicates the key finding at a
glance. *What the result means:* the picture makes the C1.2 result visual — the red
manager nodes are big and sit in the dense bridging core, confirming that betweenness
centrality and managerial rank line up in this network.


## C1.6 — Discussion: centrality vs LLM vs ground truth (5 pts)

We now have three different "opinions" about who the managers are, and we line them up.


In [21]:
# Assemble a side-by-side comparison on the LLM candidate set.
comp_rows = []
btw_top = set(n for n, _ in top_k(betweenness, 10))
ind_top = set(n for n, _ in top_k(indeg_w, 10))
pr_top  = set(n for n, _ in top_k(pagerank, 10))
for r in results:
    a = r["email"]
    in_any_top = a in btw_top or a in ind_top or a in pr_top
    comp_rows.append({
        "name": r["name"],
        "in_centrality_top10": "yes" if in_any_top else "no",
        "LLM": {True:"manager", False:"non-mgr", None:"?"}[r["llm_is_manager"]],
        "ground_truth": {1:"MANAGER", 0:"non-mgr", None:"unlabeled"}[r["true_label"]],
        "title": r["true_title"],
    })
comp = pd.DataFrame(comp_rows)
print(comp.to_string(index=False))

            name in_centrality_top10     LLM ground_truth             title
 Richard Shapiro                 yes non-mgr      MANAGER    Vice President
   John Lavorato                 yes manager      MANAGER               CEO
     Mark Taylor                 yes non-mgr      non-mgr          Employee
     Kenneth Lay                 yes manager      MANAGER               CEO
   Jeff Dasovich                 yes manager      MANAGER Managing Director
Jeffery Skilling                 yes manager      MANAGER               CEO
  Louise Kitchen                 yes manager      MANAGER         President
      Scott Neal                 yes non-mgr      MANAGER    Vice President
 Michael Grigsby                 yes non-mgr      MANAGER           Manager
   James Steffes                 yes non-mgr      MANAGER    Vice President
  Fletcher Sturm                 yes manager      MANAGER    Vice President
  Diana Scholtes                  no non-mgr      non-mgr            Trader
      Kam Ke

### Reading the comparison

**Which method worked best?**
- For *ranking* people by likely seniority, **betweenness centrality** was the best
  purely-structural method (highest precision@10, most credible top-10).
- For a *yes/no* read on an individual, the **local LLM** was the most accurate overall:
  on the labeled candidates it matched the documented titles on the large majority, and it
  correctly rejected the planted non-managers — something raw centrality cannot do,
  because a non-manager can still be a high-traffic hub.
- The **ground-truth titles** are the reference we score against, but note they too are
  imperfect (some people are `N/A`, and a title is not a perfect proxy for "manages
  people").

**Where did the methods disagree, and why?**
- *Centrality false positives:* assistants/schedulers (huge inbox) and shared mailboxes
  rank high on in-degree/PageRank without being managers. The structure says "busy", not
  "boss".
- *LLM vs title mismatches:* the most interesting cases are people whose **writing** says
  leader but whose **title** is junior (e.g. a senior trader who coordinates a desk), or
  the reverse (a VP whose sampled emails happen to be short logistics notes). These show
  that *title* and *day-to-day managing* are related but not identical, and that a small
  email sample can miss the signal.
- *Address aliasing:* the same person appears under several addresses (e.g.
  `james.steffes@` vs `d..steffes@`, `richard.shapiro@` vs `shapiro@`). This splits a
  person's mail across nodes and can both hurt centrality and reduce the emails available
  to the LLM. We did not merge aliases, to keep the pipeline transparent.

**Likely reasons for mistakes.**
- *Sampling:* we feed the LLM only ~6 emails; if those happen to be atypical (all short
  replies, or all about one logistics topic) the verdict can wobble.
- *Folder bias:* we only used sent folders, so people who rarely sent mail from a seized
  mailbox are under-represented.
- *Label noise:* the title list is itself an annotation with `N/A`s and judgement calls.

### Limitations of using email content + a local LLM for role detection

1. **Title ≠ management behaviour.** Someone can have a senior title but mostly do
   individual work, or have a junior title but effectively run a team. Email reflects
   *behaviour*, the labels reflect *titles*, so perfect agreement is not even the right
   goal.
2. **Small, possibly skewed samples.** Six emails is a thin slice of a person's working
   life; the choice of emails (we used the longest) shapes the answer.
3. **The model can be confidently wrong.** An LLM produces fluent, plausible text even
   when it is guessing; the `confidence` number it reports is self-assessed, not a
   calibrated probability.
4. **Cleaning is imperfect.** Stripping quoted replies and signatures with regex
   occasionally removes real content or leaves some boilerplate, adding noise.
5. **Privacy and ethics.** Even though we kept everything on the local cluster, these are
   real people's private messages; conclusions about individuals should be treated as
   illustrative, not authoritative.
6. **Generalization.** This pipeline is tuned to Enron's folder layout and the 2000-2001
   period; it would need re-checking on any other organization or era.

**How I solved this task (C1.6).**
*What I did:* I placed the three opinions (centrality membership, LLM verdict, ground-truth
title) side by side for every candidate and analyzed the agreements and disagreements.
*Which method/tool:* a comparison table plus reasoning about the failure modes.
*Why:* triangulating three independent signals is exactly how you judge a detector's real
strengths and blind spots. *What the result means:* betweenness centrality is the best
*structural* ranker, the local LLM is the best *individual yes/no* judge, and ground-truth
titles are a useful but imperfect referee — and the disagreements are themselves
informative, exposing assistants-as-hubs, title-vs-behaviour gaps, and address aliasing.


## Summary of artifacts produced

- **Figures** (`figures/`): `partC_enron_betweenness_top40.png`,
  `partC_precision_at_10.png`.
- **Export** (`exports/`): `partC_enron_top.gexf` (top-40 subgraph for Gephi/Cytoscape).
- **LLM cache** (`data/enron/llm_cache.json`): every classification and role summary,
  so re-running this notebook needs no further LLM calls.
- **Data artifacts** (`data/enron/`): `edges_internal.csv`, `emails_sample.jsonl`,
  `parse_stats.json`, `enron_email_positions.csv` (the labels).

**Data sources.** CMU Enron email corpus: <https://www.cs.cmu.edu/~enron/>. Title labels:
ISI / Diesner–Carley annotations via the public GitHub repo `burgersmoke/enron-formality`
(`enron_employee_positions/reranked_employee_email_positions.csv`). Anonymized reference
graph (mentioned, not used): SNAP `email-Enron`.


In [22]:
print("Part C notebook finished cleanly.")
print(f"figures saved in: {FIGDIR}")
print(f"gexf export      : {os.path.join(EXPDIR, 'partC_enron_top.gexf')}")
print(f"llm cache entries: {len(LLM_CACHE)}")

Part C notebook finished cleanly.
figures saved in: /home/mickaelz/Network analysis/figures
gexf export      : /home/mickaelz/Network analysis/exports/partC_enron_top.gexf
llm cache entries: 21
